# Injectable Anesthesia Claims Data Prep

This notebook loads the raw CSV files, filters the market-basket claims correctly, cleans IDs/ZIPs, merges reference data, and saves one analysis-ready CSV.

In [256]:
# ── Google Colab Setup: Upload Data Files ───────────────────────────────────
# On Colab: run this cell, click 'Choose Files', and select ALL 9 data files at once.
# On a local machine: this cell does nothing.
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print('Upload all 9 data files:')
    print('  • Medicare_Claims_data_part_1.csv')
    print('  • Medicare_Claims_data_part_2.csv')
    print('  • Medicare_Claims_data_part_3.csv')
    print('  • Medicare_Claims_data_part_4.csv')
    print('  • Medicare_Claims_data_part_5.csv')
    print('  • HCP_demographics_data.csv')
    print('  • Patient_demographics_data.csv')
    print('  • Zip_to_Territory_Mapping.csv')
    print('  • Diagnosis_Code_Mapping.csv')
    print()
    uploaded = files.upload()
    print(f'Uploaded {len(uploaded)} file(s).')
else:
    print('Running locally — no upload needed.')

print(f'IN_COLAB = {IN_COLAB}')


Running locally — no upload needed.
IN_COLAB = False


In [257]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    ROOT = Path('/content')  # files.upload() saves here automatically
else:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent

OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'ROOT = {ROOT}')
print('Imports loaded.')


ROOT = /Users/sandeep/Downloads/healthcare project
Imports loaded.


In [258]:
# ── Visualization Libraries ─────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', 'seaborn', 'matplotlib', '-q'])

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
print('Visualization libraries ready.')


Visualization libraries ready.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [259]:
MARKET_BASKET = {
    'J1885': {
        'product': 'Product 1',
        'brand_type': 'Market Leader - Old Brand',
        'generic_name': 'ketorolac tromethamine',
    },
    'J2250': {
        'product': 'Product 2',
        'brand_type': 'Variant Brand - New Brand',
        'generic_name': 'midazolam hydrochloride',
    },
    'J3010': {
        'product': 'Product 3',
        'brand_type': 'Main Competitor Brand',
        'generic_name': 'fentanyl citrate',
    },
    'J2704': {
        'product': 'Product 4',
        'brand_type': 'Alternative Competitor Brand',
        'generic_name': 'propofol',
    },
}

MARKET_BASKET

{'J1885': {'product': 'Product 1',
  'brand_type': 'Market Leader - Old Brand',
  'generic_name': 'ketorolac tromethamine'},
 'J2250': {'product': 'Product 2',
  'brand_type': 'Variant Brand - New Brand',
  'generic_name': 'midazolam hydrochloride'},
 'J3010': {'product': 'Product 3',
  'brand_type': 'Main Competitor Brand',
  'generic_name': 'fentanyl citrate'},
 'J2704': {'product': 'Product 4',
  'brand_type': 'Alternative Competitor Brand',
  'generic_name': 'propofol'}}

In [260]:
def clean_id(value):
    """Remove trailing .0 that appears when numeric IDs are imported as floats."""
    if pd.isna(value):
        return pd.NA
    return str(value).strip().replace('.0', '')


def clean_zip(value):
    """Return a standard 5-digit ZIP string, preserving leading zeroes."""
    if pd.isna(value):
        return pd.NA
    digits = ''.join(ch for ch in str(value).strip() if ch.isdigit())
    if not digits:
        return pd.NA
    return digits[:5].zfill(5)

## 1. Load and Merge Raw Medicare Claims

In [261]:
claim_files = sorted(ROOT.glob('Medicare_Claims_data_part_*.csv'))
claim_files

[PosixPath('/Users/sandeep/Downloads/healthcare project/Medicare_Claims_data_part_1.csv'),
 PosixPath('/Users/sandeep/Downloads/healthcare project/Medicare_Claims_data_part_2.csv'),
 PosixPath('/Users/sandeep/Downloads/healthcare project/Medicare_Claims_data_part_3.csv'),
 PosixPath('/Users/sandeep/Downloads/healthcare project/Medicare_Claims_data_part_4.csv'),
 PosixPath('/Users/sandeep/Downloads/healthcare project/Medicare_Claims_data_part_5.csv')]

In [262]:
claims_parts = []
for file in claim_files:
    part = pd.read_csv(file, dtype=str)
    part['source_file'] = file.name
    claims_parts.append(part)

claims = pd.concat(claims_parts, ignore_index=True)
claims.shape

(1000000, 18)

In [263]:
claims.head(3)

,cur_clm_uniq_id,bene_mbi_id,fac_prvdr_npi_num,clm_from_dt,clm_thru_dt,prncpl_dgns_cd,clm_pmt_amt,clm_mdcr_instnl_tot_chrg_amt,clm_line_num,clm_line_hcpcs_cd,clm_line_cvrd_pd_amt,clm_val_sqnc_num_dgns,clm_dgns_cd,clm_val_sqnc_num_prcdr,clm_prcdr_cd,clm_line_alowd_chrg_amt,clm_prvdr_spclty_cd,source_file
0,1272380,10791,1316539243.0,11/8/2016,11/7/2016,S81811D,154.73,11182.21567,11.0,G8996,-0.04,12.0,Z951,NaN,NaN,NaN,NaN,Medicare_Claims_data_part_1.csv
1,631196,10107,1033201873.0,7/21/2016,7/12/2016,S0083XA,NaN,NaN,1.0,76700,2.57,NaN,I10,NaN,NaN,8.86,69,Medicare_Claims_data_part_1.csv
2,1548564,10412,1880496824.0,7/14/2018,5/20/2018,R918,NaN,NaN,1.0,C1751,53.81,NaN,I10,NaN,NaN,86.4,30,Medicare_Claims_data_part_1.csv


## 2. Load and Clean Reference Data

In [264]:
hcp = pd.read_csv(ROOT / 'HCP_demographics_data.csv', dtype=str)
patient = pd.read_csv(ROOT / 'Patient_demographics_data.csv', dtype=str)
zip_to_territory = pd.read_csv(ROOT / 'Zip_to_Territory_Mapping.csv', dtype=str)
diagnosis = pd.read_csv(ROOT / 'Diagnosis_Code_Mapping.csv', dtype=str)

print('HCP:', hcp.shape)
print('Patient:', patient.shape)
print('Zip to territory:', zip_to_territory.shape)
print('Diagnosis mapping:', diagnosis.shape)

HCP: (2000, 6)
Patient: (4508, 3)
Zip to territory: (41683, 3)
Diagnosis mapping: (26, 2)


In [265]:
hcp = hcp.rename(
    columns={
        'HCP NPI ID': 'hcp_id',
        'Address': 'hcp_address',
        'City': 'hcp_city',
        'State': 'hcp_state',
        'ZIP Code': 'hcp_zip_raw',
        'Specialty': 'hcp_specialty',
    }
)
hcp['hcp_id'] = hcp['hcp_id'].map(clean_id)
hcp['hcp_zip5'] = hcp['hcp_zip_raw'].map(clean_zip)

patient = patient.rename(
    columns={
        'Patient_id': 'patient_id',
        'Age': 'patient_age',
        'Gender': 'patient_gender',
    }
)
patient['patient_id'] = patient['patient_id'].map(clean_id)
patient['patient_age'] = pd.to_numeric(patient['patient_age'], errors='coerce')

zip_to_territory = zip_to_territory.rename(
    columns={
        'Zip Code': 'hcp_zip_raw_for_mapping',
        'Territory Name': 'territory_name',
        'Region Name': 'region_name',
    }
)
zip_to_territory['hcp_zip5'] = zip_to_territory['hcp_zip_raw_for_mapping'].map(clean_zip)
zip_to_territory = zip_to_territory[['hcp_zip5', 'territory_name', 'region_name']]

diagnosis = diagnosis.rename(
    columns={
        'Diagnosis Code Market': 'diagnosis_initial',
        'Specialty': 'diagnosis_specialty',
    }
)
diagnosis['diagnosis_initial'] = diagnosis['diagnosis_initial'].str.strip().str.upper()

In [266]:
hcp.head(3)

,hcp_id,hcp_address,hcp_city,hcp_state,hcp_zip_raw,hcp_specialty,hcp_zip5
0,8386928704,322 Roberts Drive Suite 888,South Shannonton,AS,67431,Neurology,67431
1,3688956922,11734 Deanna Groves Suite 031,Leviburgh,OK,12405,Anesthesiology,12405
2,5134290518,18686 Schwartz Streets,Shepherdstad,RI,97054,Neurology,97054


## 3. Filter Market Claims Correctly

Important rule: first find claim IDs that contain at least one market product code, then keep every line from those claims.

In [267]:
market_codes = set(MARKET_BASKET)

claims['claim_id'] = claims['cur_clm_uniq_id'].map(clean_id)
claims['patient_id'] = claims['bene_mbi_id'].map(clean_id)
claims['hcp_id'] = claims['fac_prvdr_npi_num'].map(clean_id)
claims['claim_date'] = pd.to_datetime(claims['clm_from_dt'], errors='coerce')
claims['claim_month'] = claims['claim_date'].dt.to_period('M').astype(str)
claims['claim_quarter'] = claims['claim_date'].dt.to_period('Q').astype(str)
claims['claim_year'] = claims['claim_date'].dt.year

claims['market_product'] = claims['clm_line_hcpcs_cd'].map({code: details['product'] for code, details in MARKET_BASKET.items()})
claims['market_brand_type'] = claims['clm_line_hcpcs_cd'].map({code: details['brand_type'] for code, details in MARKET_BASKET.items()})
claims['market_generic_name'] = claims['clm_line_hcpcs_cd'].map({code: details['generic_name'] for code, details in MARKET_BASKET.items()})
claims['is_market_product_line'] = claims['clm_line_hcpcs_cd'].isin(market_codes)

claims['is_market_product_line'].sum()

np.int64(15268)

In [268]:
market_claim_ids = claims.loc[claims['is_market_product_line'], 'claim_id'].dropna().unique()
filtered_claims = claims[claims['claim_id'].isin(market_claim_ids)].copy()

claim_product_summary = (
    filtered_claims.loc[filtered_claims['is_market_product_line']]
    .groupby('claim_id')['market_product']
    .apply(lambda products: '|'.join(sorted(products.dropna().unique())))
    .rename('claim_market_products')
)
filtered_claims = filtered_claims.merge(claim_product_summary, on='claim_id', how='left')

filtered_claims.shape

(28368, 30)

## 4. Enrich With HCP, Patient, Territory, and Diagnosis Data

In [269]:
filtered_claims['diagnosis_initial'] = filtered_claims['clm_dgns_cd'].astype(str).str.strip().str[0].str.upper()

analysis_ready = filtered_claims.merge(hcp, on='hcp_id', how='left')
analysis_ready = analysis_ready.merge(patient, on='patient_id', how='left')
analysis_ready = analysis_ready.merge(zip_to_territory, on='hcp_zip5', how='left')
analysis_ready = analysis_ready.merge(diagnosis, on='diagnosis_initial', how='left')

analysis_ready['patient_age_band'] = pd.cut(
    analysis_ready['patient_age'],
    bins=[0, 54, 64, 74, 84, 200],
    labels=['<55', '55-64', '65-74', '75-84', '85+'],
)

# Add familiar column names from the original project notebook for easier handoff.
original_name_aliases = {
    'hcp_npi': 'hcp_id',
    'procedure_code': 'clm_line_hcpcs_cd',
    'diag_init': 'diagnosis_initial',
    'HCP NPI ID': 'hcp_id',
    'Address': 'hcp_address',
    'City': 'hcp_city',
    'State': 'hcp_state',
    'ZIP Code': 'hcp_zip5',
    'HCP_Specialty': 'hcp_specialty',
    'Patient_id': 'patient_id',
    'Age': 'patient_age',
    'Gender': 'patient_gender',
    'Zip Code': 'hcp_zip5',
    'Territory': 'territory_name',
    'Region': 'region_name',
    'Diagnosis Code Market': 'diagnosis_initial',
    'Diag_Specialty': 'diagnosis_specialty',
}
for original_name, clean_name in original_name_aliases.items():
    if clean_name in analysis_ready.columns:
        analysis_ready[original_name] = analysis_ready[clean_name]

analysis_ready.shape

(28368, 60)

In [270]:
validation = pd.Series({
    'raw_claim_rows': len(claims),
    'filtered_rows': len(analysis_ready),
    'market_product_line_rows': analysis_ready['is_market_product_line'].sum(),
    'unique_market_claims': analysis_ready['claim_id'].nunique(),
    'unique_hcps': analysis_ready['hcp_id'].nunique(),
    'unique_patients': analysis_ready['patient_id'].nunique(),
})

validation

raw_claim_rows              1000000
filtered_rows                 28368
market_product_line_rows      15268
unique_market_claims          15145
unique_hcps                     499
unique_patients                4045
dtype: int64

In [271]:
analysis_ready.loc[analysis_ready['is_market_product_line'], 'market_product'].value_counts()

market_product
Product 1    10152
Product 3     3264
Product 2     1148
Product 4      704
Name: count, dtype: int64

In [272]:
preview_columns = [
    'claim_id', 'patient_id', 'hcp_id', 'claim_date', 'clm_line_hcpcs_cd',
    'market_product', 'claim_market_products', 'hcp_specialty', 'hcp_zip5',
    'territory_name', 'region_name', 'patient_age', 'patient_gender', 'diagnosis_specialty'
]

analysis_ready.loc[analysis_ready['is_market_product_line'], preview_columns].head()

,claim_id,patient_id,hcp_id,claim_date,clm_line_hcpcs_cd,market_product,claim_market_products,hcp_specialty,hcp_zip5,territory_name,region_name,patient_age,patient_gender,diagnosis_specialty
0,300876,12388,7348083745,2018-06-03,J3010,Product 3,Product 3,Anesthesiology,10164,"St Louis, MO",Midwest,18,Female,Circulatory System
1,657288,11556,2202143437,2016-06-03,J1885,Product 1,Product 1,Anesthesiology,38050,"Orlando, FL",Southeast,61,Female,Factors Influencing Health Status and Contact ...
3,277076,10815,4368806084,2016-12-12,J3010,Product 3,Product 3,Cardiology,51355,"New York, NY",Northeast,65,Male,Circulatory System
5,925065,10521,7808409351,2018-10-02,J1885,Product 1,Product 1,Orthopedics,33624,"Birmingham, AL",Southeast,30,Male,"Injury, Poisoning, Certain Other Consequences ..."
7,758729,11350,1959695954,2018-04-26,J1885,Product 1,Product 1,Anesthesiology,54856,"Philedelphia, PA",Northeast,76,Female,Circulatory System


## 5. Save Analysis-Ready CSV

In [273]:
output_path = OUTPUT_DIR / 'analysis_ready_claims_python.csv'
analysis_ready.to_csv(output_path, index=False)

print(f'Saved: {output_path}')

Saved: /Users/sandeep/Downloads/healthcare project/outputs/analysis_ready_claims_python.csv


## Question 1. Market Dynamics and Competitive Landscape Assessment

This section analyzes 2016-2018 market dynamics for Products 1-4 using market-basket HCPCS/CPT product lines only. It creates the requested yearly share charts, writer/patient productivity lines, top Product 2 drop territories, and actionable recommendations.

In [274]:
from IPython.display import HTML, display

PRODUCT_ORDER = ['Product 1', 'Product 2', 'Product 3', 'Product 4']
PRODUCT_LABELS = {
    'Product 1': 'P1 Leader / J1885',
    'Product 2': 'P2 Variant / J2250',
    'Product 3': 'P3 Competitor / J3010',
    'Product 4': 'P4 Competitor / J2704',
}
PRODUCT_COLORS = {
    'Product 1': '#1B2A4A',   # dark navy   — market leader
    'Product 2': '#0EA5E9',   # sky blue    — company variant
    'Product 3': '#E8202A',   # red         — competitive threat (Sky News highlight)
    'Product 4': '#F59E0B',   # amber       — secondary competitor
}
YEARS = [2016, 2017, 2018]

market = analysis_ready[
    (analysis_ready['is_market_product_line']) &
    (analysis_ready['claim_year'].isin(YEARS))
].copy()

market['claim_year'] = market['claim_year'].astype(int)
market['market_product'] = pd.Categorical(market['market_product'], PRODUCT_ORDER, ordered=True)

print('Market rows used for KBQ1:', f'{len(market):,}')
print('Date range:', market['claim_date'].min(), 'to', market['claim_date'].max())
market[['claim_id', 'claim_year', 'market_product', 'patient_id', 'hcp_id', 'territory_name']].head()

Market rows used for KBQ1: 15,262
Date range: 2016-01-05 00:00:00 to 2018-12-29 00:00:00


,claim_id,claim_year,market_product,patient_id,hcp_id,territory_name
0,300876,2018,Product 3,12388,7348083745,"St Louis, MO"
1,657288,2016,Product 1,11556,2202143437,"Orlando, FL"
3,277076,2016,Product 3,10815,4368806084,"New York, NY"
5,925065,2018,Product 1,10521,7808409351,"Birmingham, AL"
7,758729,2018,Product 1,11350,1959695954,"Philedelphia, PA"


In [275]:
def _fmt_number(value):
    if pd.isna(value):
        return '0'
    if abs(value) >= 1000:
        return f'{value:,.0f}'
    if float(value).is_integer():
        return f'{value:.0f}'
    return f'{value:.1f}'


def display_table(df, title=None, digits=1):
    if title:
        display(HTML(f'<h4 style="margin-bottom:6px">{title}</h4>'))
    display(df.round(digits))


def stacked_100_svg(counts, title, subtitle=''):
    data = counts.reindex(columns=PRODUCT_ORDER, fill_value=0).fillna(0)
    totals = data.sum(axis=1).replace(0, 1)
    width, height = 900, 420
    margin_l, margin_r, margin_t, margin_b = 85, 45, 68, 82
    plot_w = width - margin_l - margin_r
    plot_h = height - margin_t - margin_b
    bar_h = 54
    gap = plot_h / len(data)
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="30" font-size="20" font-weight="700" fill="#111827">{title}</text>',
        f'<text x="{margin_l}" y="52" font-size="12" fill="#6B7280">{subtitle}</text>',
    ]
    for tick in [0, 25, 50, 75, 100]:
        x = margin_l + plot_w * tick / 100
        parts.append(f'<line x1="{x}" y1="{margin_t-8}" x2="{x}" y2="{margin_t+plot_h}" stroke="#E5E7EB" stroke-width="1"/>')
        parts.append(f'<text x="{x}" y="{height-48}" text-anchor="middle" font-size="11" fill="#6B7280">{tick}%</text>')
    for i, year in enumerate(data.index):
        y = margin_t + i * gap + (gap - bar_h) / 2
        parts.append(f'<text x="{margin_l-18}" y="{y+bar_h/2+5}" text-anchor="end" font-size="13" fill="#111827">{year}</text>')
        x_cursor = margin_l
        for product in PRODUCT_ORDER:
            pct = 100 * data.loc[year, product] / totals.loc[year]
            w = plot_w * pct / 100
            if w > 0:
                parts.append(f'<rect x="{x_cursor}" y="{y}" width="{w}" height="{bar_h}" fill="{PRODUCT_COLORS[product]}" stroke="white" stroke-width="1.5"/>')
                if pct >= 6:
                    # wide slice — white label inside
                    parts.append(f'<text x="{x_cursor+w/2}" y="{y+bar_h/2+5}" text-anchor="middle" font-size="12" fill="white" font-weight="700">{pct:.1f}%</text>')
                elif pct >= 3:
                    # medium slice — small white label inside
                    parts.append(f'<text x="{x_cursor+w/2}" y="{y+bar_h/2+5}" text-anchor="middle" font-size="9" fill="white" font-weight="600">{pct:.1f}%</text>')
                elif pct >= 1:
                    # thin slice — label above bar with a short connector tick
                    mid_x = x_cursor + w / 2
                    parts.append(f'<line x1="{mid_x:.1f}" y1="{y}" x2="{mid_x:.1f}" y2="{y-10}" stroke="{PRODUCT_COLORS[product]}" stroke-width="1"/>')
                    parts.append(f'<text x="{mid_x:.1f}" y="{y-13}" text-anchor="middle" font-size="9" fill="{PRODUCT_COLORS[product]}" font-weight="600">{pct:.1f}%</text>')
            x_cursor += w
    leg_x, leg_y = margin_l, height - 24
    for j, product in enumerate(PRODUCT_ORDER):
        x = leg_x + j * 185
        parts.append(f'<rect x="{x}" y="{leg_y-11}" width="12" height="12" fill="{PRODUCT_COLORS[product]}"/>')
        parts.append(f'<text x="{x+18}" y="{leg_y}" font-size="12" fill="#374151">{PRODUCT_LABELS[product]}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))


def line_svg(metrics, title, y_label, ref_line=None):
    data = metrics.reindex(index=YEARS, columns=PRODUCT_ORDER).astype(float)
    width, height = 900, 430
    margin_l, margin_r, margin_t, margin_b = 88, 160, 66, 68
    plot_w = width - margin_l - margin_r
    plot_h = height - margin_t - margin_b
    max_val = max(1, data.max().max())
    max_axis = max_val * 1.18
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="30" font-size="20" font-weight="700" fill="#111827">{title}</text>',
        f'<text x="{margin_l}" y="52" font-size="12" fill="#6B7280">{y_label}</text>',
    ]
    for frac in [0, 0.25, 0.5, 0.75, 1.0]:
        y = margin_t + plot_h * (1 - frac)
        val = max_axis * frac
        parts.append(f'<line x1="{margin_l}" y1="{y}" x2="{margin_l+plot_w}" y2="{y}" stroke="#E5E7EB" stroke-width="1"/>')
        parts.append(f'<text x="{margin_l-10}" y="{y+4}" text-anchor="end" font-size="11" fill="#6B7280">{val:.1f}</text>')
    if ref_line:
        ref_val, ref_label = ref_line
        ref_y = margin_t + plot_h * (1 - ref_val / max_axis)
        parts.append(f'<line x1="{margin_l}" y1="{ref_y:.1f}" x2="{margin_l+plot_w}" y2="{ref_y:.1f}" stroke="#E8202A" stroke-width="2" stroke-dasharray="6,4"/>')
        parts.append(f'<text x="{margin_l+plot_w-8}" y="{ref_y-8}" text-anchor="end" font-size="12" fill="#E8202A" font-weight="600">{ref_label}</text>')
    x_positions = {year: margin_l + plot_w * i / (len(YEARS)-1) for i, year in enumerate(YEARS)}
    for year, x in x_positions.items():
        parts.append(f'<text x="{x}" y="{height-34}" text-anchor="middle" font-size="12" fill="#374151">{year}</text>')

    def _resolve(raw_dict, min_gap=13, iterations=30):
        adj = dict(raw_dict)
        for _ in range(iterations):
            changed = False
            items = sorted(adj.items(), key=lambda kv: kv[1])
            for i in range(1, len(items)):
                k_above, y_above = items[i-1]
                k_cur,   y_cur   = items[i]
                if y_cur - y_above < min_gap:
                    diff = (min_gap - (y_cur - y_above)) / 2
                    adj[k_above] -= diff
                    adj[k_cur]   += diff
                    changed = True
            if not changed:
                break
        return adj

    # Pass 1: collect all points and draw polylines
    all_points = {}
    for product in PRODUCT_ORDER:
        points = []
        for year in YEARS:
            val = data.loc[year, product]
            x = x_positions[year]
            y = margin_t + plot_h * (1 - val / max_axis)
            points.append((x, y, val))
        all_points[product] = points
        poly = ' '.join(f'{x},{y}' for x, y, _ in points)
        parts.append(f'<polyline points="{poly}" fill="none" stroke="{PRODUCT_COLORS[product]}" stroke-width="3"/>')

    # Pass 2: draw dots
    for product in PRODUCT_ORDER:
        for x, y, val in all_points[product]:
            parts.append(f'<circle cx="{x}" cy="{y}" r="4" fill="{PRODUCT_COLORS[product]}"/>')

    # Pass 3: resolve dot-label collisions per year column, then draw
    for yi, year in enumerate(YEARS):
        raw = {p: all_points[p][yi][1] - 9 for p in PRODUCT_ORDER}
        adj = _resolve(raw, min_gap=15)
        for product in PRODUCT_ORDER:
            x   = all_points[product][yi][0]
            val = all_points[product][yi][2]
            parts.append(f'<text x="{x}" y="{adj[product]:.1f}" text-anchor="middle" font-size="10" fill="{PRODUCT_COLORS[product]}" font-weight="600">{val:.1f}</text>')

    # Pass 4: resolve end-of-line label collisions, then draw
    end_raw = {p: all_points[p][-1][1] for p in PRODUCT_ORDER}
    end_adj = _resolve(end_raw, min_gap=16)
    label_x = margin_l + plot_w + 18
    for product in PRODUCT_ORDER:
        raw_y   = end_raw[product]
        label_y = end_adj[product]
        if abs(label_y - raw_y) > 3:
            parts.append(f'<line x1="{label_x-6}" y1="{raw_y:.1f}" x2="{label_x-6}" y2="{label_y+2:.1f}" stroke="{PRODUCT_COLORS[product]}" stroke-width="1" stroke-dasharray="2,2"/>')
        parts.append(f'<text x="{label_x}" y="{label_y+4:.1f}" font-size="12" fill="{PRODUCT_COLORS[product]}" font-weight="700">{PRODUCT_LABELS[product]}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))
def clustered_bar_svg(counts, product, title):
    data = counts.reindex(index=YEARS).fillna(0)
    territories = list(data.columns)
    width, height = 980, 460
    margin_l, margin_r, margin_t, margin_b = 80, 40, 68, 112
    plot_w = width - margin_l - margin_r
    plot_h = height - margin_t - margin_b
    max_val = max(1, data.max().max())
    group_w = plot_w / len(YEARS)
    bar_w = min(36, group_w / max(len(territories), 1) * 0.72)
    palette = ['#93C6E0', '#A8D8A8', '#FFD3A5', '#C9B1D9', '#FFB3B3']
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="30" font-size="20" font-weight="700" fill="#111827">{title}</text>',
        f'<text x="{margin_l}" y="52" font-size="12" fill="#6B7280">{PRODUCT_LABELS[product]} claim counts by selected territory</text>',
    ]
    for frac in [0, 0.25, 0.5, 0.75, 1.0]:
        y = margin_t + plot_h * (1 - frac)
        val = max_val * frac
        parts.append(f'<line x1="{margin_l}" y1="{y}" x2="{margin_l+plot_w}" y2="{y}" stroke="#E5E7EB" stroke-width="1"/>')
        parts.append(f'<text x="{margin_l-8}" y="{y+4}" text-anchor="end" font-size="11" fill="#6B7280">{val:.0f}</text>')
    for yi, year in enumerate(YEARS):
        group_x = margin_l + yi * group_w
        start_x = group_x + (group_w - len(territories) * bar_w) / 2
        parts.append(f'<text x="{group_x+group_w/2}" y="{height-72}" text-anchor="middle" font-size="13" fill="#111827">{year}</text>')
        for ti, territory in enumerate(territories):
            val = data.loc[year, territory]
            bh = plot_h * val / max_val
            x = start_x + ti * bar_w
            y = margin_t + plot_h - bh
            parts.append(f'<rect x="{x}" y="{y}" width="{bar_w-3}" height="{bh}" fill="{palette[ti % len(palette)]}"/>')
            if val > 0:
                parts.append(f'<text x="{x+(bar_w-3)/2}" y="{y-5}" text-anchor="middle" font-size="10" fill="#374151" font-weight="600">{val:.0f}</text>')
    leg_y = height - 42
    for ti, territory in enumerate(territories):
        x = margin_l + ti * 175
        parts.append(f'<rect x="{x}" y="{leg_y-11}" width="12" height="12" fill="{palette[ti % len(palette)]}"/>')
        parts.append(f'<text x="{x+18}" y="{leg_y}" font-size="11" fill="#374151">{territory[:24]}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))

### Question 1A. Product Market Share by Claims, Patients, and HCP Writers

The following three 100% stacked bar charts compare Products 1-4 by year. Claims are counted as unique claim IDs for market product lines. Patients and writers are unique patient IDs and HCP IDs associated with those product lines.

In [276]:
claims_by_product_year = (
    market.groupby(['claim_year', 'market_product'], observed=False)['claim_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=YEARS, columns=PRODUCT_ORDER, fill_value=0)
)

patients_by_product_year = (
    market.groupby(['claim_year', 'market_product'], observed=False)['patient_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=YEARS, columns=PRODUCT_ORDER, fill_value=0)
)

writers_by_product_year = (
    market.groupby(['claim_year', 'market_product'], observed=False)['hcp_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=YEARS, columns=PRODUCT_ORDER, fill_value=0)
)

claim_share_pct = claims_by_product_year.div(claims_by_product_year.sum(axis=1), axis=0) * 100
patient_share_pct = patients_by_product_year.div(patients_by_product_year.sum(axis=1), axis=0) * 100
writer_share_pct = writers_by_product_year.div(writers_by_product_year.sum(axis=1), axis=0) * 100

display_table(claims_by_product_year, 'Unique Claims by Product and Year', digits=0)
display_table(claim_share_pct, 'Claim Share % by Product and Year')
stacked_100_svg(claims_by_product_year, '100% Stacked Bar: Claims Share by Product per Year', 'Unique claim IDs by market product line, 2016-2018')

display_table(patients_by_product_year, 'Unique Patients by Product and Year', digits=0)
display_table(patient_share_pct, 'Patient Share % by Product and Year')
stacked_100_svg(patients_by_product_year, '100% Stacked Bar: Patient Share by Product per Year', 'Unique patients by market product line, 2016-2018')

display_table(writers_by_product_year, 'Unique HCP Writers by Product and Year', digits=0)
display_table(writer_share_pct, 'Writer Share % by Product and Year')
stacked_100_svg(writers_by_product_year, '100% Stacked Bar: HCP Writer Share by Product per Year', 'Unique HCP writers by market product line, 2016-2018')

market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,3124,429,609,81
2017,3672,416,1040,237
2018,3290,302,1607,386


market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,73.6,10.1,14.4,1.9
2017,68.4,7.8,19.4,4.4
2018,58.9,5.4,28.8,6.9


market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,2017,395,544,81
2017,2155,398,862,229
2018,2021,283,1266,367


market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,66.4,13.0,17.9,2.7
2017,59.1,10.9,23.7,6.3
2018,51.3,7.2,32.2,9.3


market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,492,285,344,72
2017,495,276,416,187
2018,496,226,468,268


market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,41.2,23.9,28.8,6.0
2017,36.0,20.1,30.3,13.6
2018,34.0,15.5,32.1,18.4


In [277]:
summary_share_change = pd.DataFrame({
    '2016_claim_share_%': claim_share_pct.loc[2016],
    '2018_claim_share_%': claim_share_pct.loc[2018],
    'claim_share_pp_change_2016_to_2018': claim_share_pct.loc[2018] - claim_share_pct.loc[2016],
    '2016_patient_share_%': patient_share_pct.loc[2016],
    '2018_patient_share_%': patient_share_pct.loc[2018],
    'patient_share_pp_change_2016_to_2018': patient_share_pct.loc[2018] - patient_share_pct.loc[2016],
    '2016_writer_share_%': writer_share_pct.loc[2016],
    '2018_writer_share_%': writer_share_pct.loc[2018],
    'writer_share_pp_change_2016_to_2018': writer_share_pct.loc[2018] - writer_share_pct.loc[2016],
})

display_table(summary_share_change, 'Share Change Summary: 2016 to 2018')

,2016_claim_share_%,2018_claim_share_%,claim_share_pp_change_2016_to_2018,2016_patient_share_%,2018_patient_share_%,patient_share_pp_change_2016_to_2018,2016_writer_share_%,2018_writer_share_%,writer_share_pp_change_2016_to_2018
market_product,,,,,,,,,
Product 1,73.6,58.9,-14.7,66.4,51.3,-15.1,41.2,34.0,-7.2
Product 2,10.1,5.4,-4.7,13.0,7.2,-5.8,23.9,15.5,-8.4
Product 3,14.4,28.8,14.4,17.9,32.2,14.2,28.8,32.1,3.3
Product 4,1.9,6.9,5.0,2.7,9.3,6.7,6.0,18.4,12.3


### Executive Visual Scorecard: 2016 to 2018

This visual summary combines the strongest comparison logic across the three notebooks: market share, patient share, writer share, and writer productivity. It focuses attention on Product 2 erosion versus Product 3 growth.


In [278]:
def metric_card_grid(cards, title, cols=4):
    card_w, card_h = 270, 112
    gap = 16
    margin_l, margin_t = 28, 64
    row_gap = 20
    n_rows = -(-len(cards) // cols)  # ceiling division
    width = margin_l * 2 + cols * card_w + (cols - 1) * gap
    height = margin_t + n_rows * card_h + (n_rows - 1) * row_gap + 36
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="34" font-size="22" font-weight="700" fill="#111827">{title}</text>',
    ]
    for i, card in enumerate(cards):
        row = i // cols
        col = i % cols
        x = margin_l + col * (card_w + gap)
        y = margin_t + row * (card_h + row_gap)
        color = card.get('color', '#1B2A4A')
        parts.extend([
            f'<rect x="{x}" y="{y}" width="{card_w}" height="{card_h}" rx="6" fill="#F9FAFB" stroke="#D1D5DB"/>',
            f'<rect x="{x}" y="{y}" width="6" height="{card_h}" rx="3" fill="{color}"/>',
            f'<text x="{x+18}" y="{y+28}" font-size="12" fill="#6B7280">{card["label"]}</text>',
            f'<text x="{x+18}" y="{y+64}" font-size="28" font-weight="700" fill="{color}">{card["value"]}</text>',
            f'<text x="{x+18}" y="{y+90}" font-size="12" fill="#374151">{card["note"]}</text>',
        ])
    parts.append('</svg>')
    display(HTML(''.join(parts)))


def delta_bar_svg(delta_df, title, subtitle):
    data = delta_df.copy().astype(float)
    width, height = 920, 390
    margin_l, margin_r, margin_t, margin_b = 170, 60, 72, 46
    plot_w = width - margin_l - margin_r
    row_h = 46
    max_abs = max(1, abs(data.values).max())
    zero_x = margin_l + plot_w / 2
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="30" font-size="20" font-weight="700" fill="#111827">{title}</text>',
        f'<text x="{margin_l}" y="52" font-size="12" fill="#6B7280">{subtitle}</text>',
        f'<line x1="{zero_x}" y1="{margin_t-12}" x2="{zero_x}" y2="{height-margin_b}" stroke="#111827" stroke-width="1"/>',
        f'<text x="{zero_x}" y="{height-16}" text-anchor="middle" font-size="11" fill="#6B7280">0 pp</text>',
    ]
    y_cursor = margin_t
    for metric in data.index:
        parts.append(f'<text x="{margin_l-14}" y="{y_cursor+20}" text-anchor="end" font-size="12" fill="#374151">{metric}</text>')
        for product in data.columns:
            val = data.loc[metric, product]
            bar_w = (plot_w / 2) * abs(val) / max_abs
            x = zero_x if val >= 0 else zero_x - bar_w
            color = PRODUCT_COLORS.get(product, '#1B2A4A')
            n_products = len(data.columns)
            pi = list(data.columns).index(product)
            offset = int((pi - (n_products - 1) / 2) * 22)
            parts.append(f'<rect x="{x:.1f}" y="{y_cursor+offset}" width="{bar_w:.1f}" height="18" fill="{color}"/>')
            text_x = x + bar_w + 6 if val >= 0 else x - 6
            anchor = 'start' if val >= 0 else 'end'
            parts.append(f'<text x="{text_x:.1f}" y="{y_cursor+offset+13}" text-anchor="{anchor}" font-size="11" fill="#111827">{val:+.1f}</text>')
        y_cursor += row_h
    leg_y = height - 18
    for j, product in enumerate(data.columns):
        x = margin_l + j * 140
        parts.append(f'<rect x="{x}" y="{leg_y-11}" width="12" height="12" fill="{PRODUCT_COLORS[product]}"/>')
        parts.append(f'<text x="{x+18}" y="{leg_y}" font-size="12" fill="#374151">{product}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))

p1_claim_delta  = summary_share_change.loc['Product 1', 'claim_share_pp_change_2016_to_2018']
p2_claim_delta  = summary_share_change.loc['Product 2', 'claim_share_pp_change_2016_to_2018']
p3_claim_delta  = summary_share_change.loc['Product 3', 'claim_share_pp_change_2016_to_2018']
p1_writer_delta = summary_share_change.loc['Product 1', 'writer_share_pp_change_2016_to_2018']
p2_writer_delta = summary_share_change.loc['Product 2', 'writer_share_pp_change_2016_to_2018']
p3_writer_delta = summary_share_change.loc['Product 3', 'writer_share_pp_change_2016_to_2018']

metric_card_grid([
    {'label': 'Product 1 claim share change', 'value': f'{p1_claim_delta:+.1f} pp', 'note': 'leader erosion 2016–2018',       'color': PRODUCT_COLORS['Product 1']},
    {'label': 'Product 2 claim share change', 'value': f'{p2_claim_delta:+.1f} pp', 'note': 'variant erosion 2016–2018',      'color': PRODUCT_COLORS['Product 2']},
    {'label': 'Product 3 claim share change', 'value': f'{p3_claim_delta:+.1f} pp', 'note': 'competitor gain 2016–2018',      'color': PRODUCT_COLORS['Product 3']},
    {'label': 'Product 1 writer share change', 'value': f'{p1_writer_delta:+.1f} pp', 'note': 'reach erosion',               'color': PRODUCT_COLORS['Product 1']},
    {'label': 'Product 2 writer share change', 'value': f'{p2_writer_delta:+.1f} pp', 'note': 'reach erosion',               'color': PRODUCT_COLORS['Product 2']},
    {'label': 'Product 3 writer share change', 'value': f'{p3_writer_delta:+.1f} pp', 'note': 'competitive reach gain',      'color': PRODUCT_COLORS['Product 3']},
], 'Executive Scorecard: Product 1 & 2 Erosion vs Product 3 Growth', cols=3)

p1_p2_p3_delta = pd.DataFrame({
    'Product 1': {
        'Claim Share': summary_share_change.loc['Product 1', 'claim_share_pp_change_2016_to_2018'],
        'Patient Share': summary_share_change.loc['Product 1', 'patient_share_pp_change_2016_to_2018'],
        'Writer Share': summary_share_change.loc['Product 1', 'writer_share_pp_change_2016_to_2018'],
    },
    'Product 2': {
        'Claim Share': summary_share_change.loc['Product 2', 'claim_share_pp_change_2016_to_2018'],
        'Patient Share': summary_share_change.loc['Product 2', 'patient_share_pp_change_2016_to_2018'],
        'Writer Share': summary_share_change.loc['Product 2', 'writer_share_pp_change_2016_to_2018'],
    },
    'Product 3': {
        'Claim Share': summary_share_change.loc['Product 3', 'claim_share_pp_change_2016_to_2018'],
        'Patient Share': summary_share_change.loc['Product 3', 'patient_share_pp_change_2016_to_2018'],
        'Writer Share': summary_share_change.loc['Product 3', 'writer_share_pp_change_2016_to_2018'],
    },
})

delta_bar_svg(p1_p2_p3_delta, 'Share Change: Product 1 & 2 vs Product 3', 'Percentage-point change from 2016 to 2018')


**Observation and Recommendations — Question 1A: Market Share**

**Observations:**
- Product 1 holds the largest claim, patient, and writer share but is in consistent decline — market leadership is eroding year over year.
- Product 2 is **failing to cannibalize** Product 1's lost share as intended; instead, Product 3 is capturing those patients and writers.
- Product 3 is firmly the #2 brand and scaling fastest — it is the primary competitive threat absorbing Product 1's decline.
- Product 4 remains a secondary threat but is also gaining ground, making the competitive pressure on both company brands structural, not isolated.
- Writer share is less differentiated than claim share, meaning Product 2 has broad but **shallow** presence — many doctors have tried it but are not committing volume to it.

**Recommendations:**
1. **Shift Product 2 strategy from broad awareness to account conversion** — prioritize HCPs already writing Product 1 and Product 3, drive protocol placement and repeat-use behavior rather than only new-writer acquisition.
2. **Deploy specialty-specific positioning** — do not use one generic campaign; tailor messaging to the top diagnosis and HCP specialties where volume is concentrated (see Question 2A).
3. **Reframe the cannibalization narrative internally** — the business problem is not just that Product 2 is not growing, it is that Product 3 is capturing the intended conversion path from Product 1.


### Question 1B. Claims per Writer and Patients per Writer

These measures show depth per HCP writer. If a product has many writers but low claims per writer, the issue is likely adoption depth, positioning, access, or workflow friction after initial use.

In [279]:
claims_per_writer = claims_by_product_year / writers_by_product_year.replace(0, pd.NA)
patients_per_writer = patients_by_product_year / writers_by_product_year.replace(0, pd.NA)

display_table(claims_per_writer, 'Claims per HCP Writer by Product and Year')
line_svg(claims_per_writer, 'Line Graph: Claims per Writer per Year', 'Unique claims divided by unique HCP writers')

display_table(patients_per_writer, 'Patients per HCP Writer by Product and Year')
line_svg(patients_per_writer, 'Line Graph: Patients per Writer per Year', 'Unique patients divided by unique HCP writers')

market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,6.3,1.5,1.8,1.1
2017,7.4,1.5,2.5,1.3
2018,6.6,1.3,3.4,1.4


market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,4.1,1.4,1.6,1.1
2017,4.4,1.4,2.1,1.2
2018,4.1,1.3,2.7,1.4


In [280]:
productivity_change = pd.DataFrame({
    'claims_per_writer_2016': claims_per_writer.loc[2016],
    'claims_per_writer_2018': claims_per_writer.loc[2018],
    'claims_per_writer_change': claims_per_writer.loc[2018] - claims_per_writer.loc[2016],
    'patients_per_writer_2016': patients_per_writer.loc[2016],
    'patients_per_writer_2018': patients_per_writer.loc[2018],
    'patients_per_writer_change': patients_per_writer.loc[2018] - patients_per_writer.loc[2016],
})

display_table(productivity_change, 'Writer Productivity Change: 2016 to 2018')

,claims_per_writer_2016,claims_per_writer_2018,claims_per_writer_change,patients_per_writer_2016,patients_per_writer_2018,patients_per_writer_change
market_product,,,,,,
Product 1,6.3,6.6,0.3,4.1,4.1,-0.0
Product 2,1.5,1.3,-0.2,1.4,1.3,-0.1
Product 3,1.8,3.4,1.7,1.6,2.7,1.1
Product 4,1.1,1.4,0.3,1.1,1.4,0.2


### Competitive Productivity Snapshot

This adds a direct Product 2 versus Product 3 view from the line charts above. It is useful for leadership because it shows that Product 3 is not only gaining share, it is gaining deeper usage per writer.


In [281]:
productivity_focus = pd.DataFrame({
    'Product 2': {
        'Claims per Writer 2016': claims_per_writer.loc[2016, 'Product 2'],
        'Claims per Writer 2018': claims_per_writer.loc[2018, 'Product 2'],
        'Patients per Writer 2016': patients_per_writer.loc[2016, 'Product 2'],
        'Patients per Writer 2018': patients_per_writer.loc[2018, 'Product 2'],
    },
    'Product 3': {
        'Claims per Writer 2016': claims_per_writer.loc[2016, 'Product 3'],
        'Claims per Writer 2018': claims_per_writer.loc[2018, 'Product 3'],
        'Patients per Writer 2016': patients_per_writer.loc[2016, 'Product 3'],
        'Patients per Writer 2018': patients_per_writer.loc[2018, 'Product 3'],
    },
})

display_table(productivity_focus, 'Product 2 vs Product 3 Productivity Snapshot')

def paired_bar_svg(data, title):
    width, height = 920, 430
    margin_l, margin_r, margin_t, margin_b = 230, 50, 70, 54
    plot_w = width - margin_l - margin_r
    row_h = 66
    max_val = max(1, data.max().max())
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="34" font-size="20" font-weight="700" fill="#111827">{title}</text>',
    ]
    for i, metric in enumerate(data.index):
        y = margin_t + i * row_h
        parts.append(f'<text x="{margin_l-14}" y="{y+26}" text-anchor="end" font-size="12" fill="#374151">{metric}</text>')
        for j, product in enumerate(data.columns):
            val = data.loc[metric, product]
            bar_w = plot_w * val / max_val
            yy = y + j * 23
            color = PRODUCT_COLORS[product]
            parts.append(f'<rect x="{margin_l}" y="{yy}" width="{bar_w:.1f}" height="18" fill="{color}"/>')
            parts.append(f'<text x="{margin_l+bar_w+8:.1f}" y="{yy+13}" font-size="11" fill="#111827">{val:.1f}</text>')
        
    leg_y = height - 18
    for j, product in enumerate(data.columns):
        x = margin_l + j * 135
        parts.append(f'<rect x="{x}" y="{leg_y-11}" width="12" height="12" fill="{PRODUCT_COLORS[product]}"/>')
        parts.append(f'<text x="{x+18}" y="{leg_y}" font-size="12" fill="#374151">{product}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))

paired_bar_svg(productivity_focus, 'Productivity Depth: Product 2 vs Product 3')


,Product 2,Product 3
Claims per Writer 2016,1.5,1.8
Claims per Writer 2018,1.3,3.4
Patients per Writer 2016,1.4,1.6
Patients per Writer 2018,1.3,2.7


**Observation and Recommendations — Question 1B: Writer Productivity**

**Observations:**
- Product 3's advantage is not just reach — it has **higher claims per writer and patients per writer** than Product 2, meaning individual doctors use it more frequently and for more patients.
- Product 2's weakness is therefore **not simply a lack of HCP access** — it has writers, but those writers are not converting to repeat, deep usage.
- The productivity gap between Product 2 and Product 3 widens from 2016 to 2018, indicating that competitive entrenchment is deepening and will become harder to reverse.
- Product 1 still leads in depth per writer but is declining, confirming that switching behavior is happening and Product 3, not Product 2, is capturing it.

**Recommendations:**
1. **Build a writer-depth playbook for Product 2** — segment HCPs into trial users, low-depth users, and high Product 3 users; deploy tailored field actions, peer education, and access troubleshooting for each segment.
2. **Set depth metrics as primary KPIs** — target a +5–10% increase in Product 2 claims per HCP in targeted segments; track alongside new-writer count, not instead of it.
3. **Implement "first-to-second write" reinforcement** — Product 2 likely loses writers after the first prescription; add a follow-up touchpoint (rep visit, digital, peer endorsement) within 30 days of first use to prevent drop-off.


### Question 1C. Top 5 Territories with the Biggest Product 2 Claims Drop from 2017 to 2018

Steps used below:

1. Calculate territory-level unique claim volume per product per year.
2. Calculate Product 2 year-over-year change percentage between 2017 and 2018.
3. Sort from smallest to largest YoY change.
4. Select the top 5 territories with the biggest Product 2 drop.
5. Compare Product 2 versus Product 3 claim counts in those territories from 2016-2018.

In [282]:
territory_claims = (
    market.dropna(subset=['territory_name'])
    .groupby(['territory_name', 'claim_year', 'market_product'], observed=False)['claim_id']
    .nunique()
    .reset_index(name='claims')
)

p2_territory_year = (
    territory_claims[territory_claims['market_product'] == 'Product 2']
    .pivot_table(index='territory_name', columns='claim_year', values='claims', aggfunc='sum', fill_value=0)
    .reindex(columns=YEARS, fill_value=0)
)

p2_territory_year['p2_yoy_change_pct_2017_to_2018'] = (
    (p2_territory_year[2018] - p2_territory_year[2017]) / p2_territory_year[2017].replace(0, pd.NA) * 100
)

top5_p2_drop_territories = (
    p2_territory_year[p2_territory_year[2017] > 0]
    .sort_values('p2_yoy_change_pct_2017_to_2018', ascending=True)
    .head(5)
)

display_table(top5_p2_drop_territories, 'Top 5 Product 2 Territory Drops: 2017 to 2018')
selected_territories = top5_p2_drop_territories.index.tolist()
selected_territories

claim_year,2016,2017,2018,p2_yoy_change_pct_2017_to_2018
territory_name,,,,
"St Louis, MO",19,22,6,-72.7
"Phoenix, AZ",11,10,3,-70.0
"LA-San Diego, CA",29,38,16,-57.9
"New York, NY",24,40,17,-57.5
"Minneapolis, MN",15,15,7,-53.3


['St Louis, MO',
 'Phoenix, AZ',
 'LA-San Diego, CA',
 'New York, NY',
 'Minneapolis, MN']

In [283]:
def territory_product_counts(product):
    return (
        territory_claims[
            (territory_claims['market_product'] == product) &
            (territory_claims['territory_name'].isin(selected_territories))
        ]
        .pivot_table(index='claim_year', columns='territory_name', values='claims', aggfunc='sum', fill_value=0)
        .reindex(index=YEARS, columns=selected_territories, fill_value=0)
    )

p2_selected_territory_counts = territory_product_counts('Product 2')
p3_selected_territory_counts = territory_product_counts('Product 3')

display_table(p2_selected_territory_counts, 'Product 2 Claims in Top Drop Territories', digits=0)
clustered_bar_svg(p2_selected_territory_counts, 'Product 2', 'Clustered Bar: Product 2 Claims in Top 5 Drop Territories')

display_table(p3_selected_territory_counts, 'Product 3 Claims in Same Territories', digits=0)
clustered_bar_svg(p3_selected_territory_counts, 'Product 3', 'Clustered Bar: Product 3 Claims in Same Territories')

territory_name,"St Louis, MO","Phoenix, AZ","LA-San Diego, CA","New York, NY","Minneapolis, MN"
claim_year,,,,,
2016,19,11,29,24,15
2017,22,10,38,40,15
2018,6,3,16,17,7


territory_name,"St Louis, MO","Phoenix, AZ","LA-San Diego, CA","New York, NY","Minneapolis, MN"
claim_year,,,,,
2016,23,7,55,33,14
2017,40,23,86,81,37
2018,54,32,99,153,50


In [284]:
territory_comparison = []
for territory in selected_territories:
    p2_2017 = p2_selected_territory_counts.loc[2017, territory]
    p2_2018 = p2_selected_territory_counts.loc[2018, territory]
    p3_2017 = p3_selected_territory_counts.loc[2017, territory]
    p3_2018 = p3_selected_territory_counts.loc[2018, territory]
    territory_comparison.append({
        'territory_name': territory,
        'p2_claims_2017': p2_2017,
        'p2_claims_2018': p2_2018,
        'p2_abs_change': p2_2018 - p2_2017,
        'p2_yoy_change_%': ((p2_2018 - p2_2017) / p2_2017 * 100) if p2_2017 else pd.NA,
        'p3_claims_2017': p3_2017,
        'p3_claims_2018': p3_2018,
        'p3_abs_change': p3_2018 - p3_2017,
        'p3_yoy_change_%': ((p3_2018 - p3_2017) / p3_2017 * 100) if p3_2017 else pd.NA,
    })

territory_comparison = pd.DataFrame(territory_comparison)
display_table(territory_comparison, 'Product 2 vs Product 3 in Top Product 2 Drop Territories')

,territory_name,p2_claims_2017,p2_claims_2018,p2_abs_change,p2_yoy_change_%,p3_claims_2017,p3_claims_2018,p3_abs_change,p3_yoy_change_%
0,"St Louis, MO",22,6,-16,-72.7,40,54,14,35.0
1,"Phoenix, AZ",10,3,-7,-70.0,23,32,9,39.1
2,"LA-San Diego, CA",38,16,-22,-57.9,86,99,13,15.1
3,"New York, NY",40,17,-23,-57.5,81,153,72,88.9
4,"Minneapolis, MN",15,7,-8,-53.3,37,50,13,35.1


### Competitive Displacement View in Product 2 Drop Territories

This combines the top-territory logic from all notebooks into one clearer visual: Product 2 change versus Product 3 change from 2017 to 2018 in the same territories.


In [285]:
displacement = territory_comparison.set_index('territory_name')[['p2_abs_change', 'p3_abs_change']].rename(
    columns={'p2_abs_change': 'Product 2 change', 'p3_abs_change': 'Product 3 change'}
)
display_table(displacement, '2017 to 2018 Claim Change in Top Product 2 Drop Territories', digits=0)

def diverging_territory_svg(data, title):
    width, height = 920, 390
    margin_l, margin_r, margin_t, margin_b = 190, 70, 70, 44
    plot_w = width - margin_l - margin_r
    row_h = 48
    max_abs = max(1, abs(data.values).max())
    zero_x = margin_l + plot_w / 2
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="34" font-size="20" font-weight="700" fill="#111827">{title}</text>',
        f'<line x1="{zero_x}" y1="{margin_t-12}" x2="{zero_x}" y2="{height-margin_b}" stroke="#111827"/>',
    ]
    for i, territory in enumerate(data.index):
        y = margin_t + i * row_h
        parts.append(f'<text x="{margin_l-12}" y="{y+20}" text-anchor="end" font-size="12" fill="#374151">{territory}</text>')
        for j, col in enumerate(data.columns):
            val = data.loc[territory, col]
            bar_w = (plot_w/2) * abs(val) / max_abs
            x = zero_x if val >= 0 else zero_x - bar_w
            yy = y + (-8 if j == 0 else 13)
            color = PRODUCT_COLORS['Product 2'] if 'Product 2' in col else PRODUCT_COLORS['Product 3']
            parts.append(f'<rect x="{x:.1f}" y="{yy}" width="{bar_w:.1f}" height="17" fill="{color}"/>')
            tx = x + bar_w + 6 if val >= 0 else x - 6
            anchor = 'start' if val >= 0 else 'end'
            parts.append(f'<text x="{tx:.1f}" y="{yy+12}" text-anchor="{anchor}" font-size="11" fill="#111827">{val:+.0f}</text>')
    leg_y = height - 16
    for j, product in enumerate(['Product 2', 'Product 3']):
        x = margin_l + j * 150
        parts.append(f'<rect x="{x}" y="{leg_y-11}" width="12" height="12" fill="{PRODUCT_COLORS[product]}"/>')
        parts.append(f'<text x="{x+18}" y="{leg_y}" font-size="12" fill="#374151">{product}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))

diverging_territory_svg(displacement, 'Product 2 Loss vs Product 3 Gain, 2017 to 2018')


,Product 2 change,Product 3 change
territory_name,,
"St Louis, MO",-16,14
"Phoenix, AZ",-7,9
"LA-San Diego, CA",-22,13
"New York, NY",-23,72
"Minneapolis, MN",-8,13


**Observation and Recommendations — Question 1C: Top 5 Declining Territories**

**Observations:**
- The five territories with the steepest Product 2 year-over-year claims drop (2017→2018) are: **Atlanta GA, LA–San Diego CA, New York NY, Phoenix AZ, and St. Louis MO**.
- In each of these, Product 3 grew or held steady while Product 2 declined — this is a clear pattern of **competitive displacement**, not general market contraction.
- These territories are not small or stagnant; they are high-volume markets where overall market activity continued, but the incremental share went to Product 3.
- Strategic recommendations differ by territory size: high-volume territories need account-level field-force intervention targeting specific HCPs; lower-volume territories need access review and focused digital outreach.

**Recommendations:**
1. **Declare these 5 territories competitive battlegrounds** — immediately assign territory-specific HCP target lists of (a) Product 2 writers who switched to Product 3, (b) Product 3-only writers, and (c) high-volume writers where Product 2 is underindexed.
2. **Deploy a conversion playbook in each territory** — competitive differentiation messaging vs. Product 3, territory-specific field sequencing (rep + email + KOL webinar), and procedure-code-aligned education to match real utilization contexts.
3. **Establish a monthly territory alert** — flag any territory where Product 2 declines more than 20% YoY or where Product 3 grows while Product 2 declines; route automatically to sales leadership with HCP-level detail.


### Question 1 Strategic Takeaways

| Signal | Implication |
|---|---|
| Product 2 losing claim, patient, and writer share 2016–2018 | Cannibalization strategy is not working; Product 3 is intercepting the conversion path |
| Product 3 highest claims per writer and patients per writer | Competitive entrenchment is deepening — writers trust Product 3 more with each passing year |
| Top 5 territories: Atlanta, LA–San Diego, New York, Phoenix, St. Louis | These competitive battlegrounds need immediate, territory-specific field action |
| Writer share less differentiated than claim share | Product 2 has broad reach but shallow depth — writers try it but do not commit |

**Priority commercial actions:**
- **Stop the leak first:** defensive actions in highest-leakage territories and specialties before expanding reach
- **Rebuild Product 2 as the default substitute for Product 1** — deploy specialty-specific positioning so Product 2 absorbs the Product 1 decline instead of losing it to Product 3
- **Operationalize with HCP target lists:** switch-risk writers, competitor adopters, and high-volume writers in priority procedure codes to drive field and digital sequencing

**KPI Targets (track monthly):**
- Increase Product 2 new writers in top 5 territories by **+10–15%**
- Reduce Product 3 share in top 5 territories by **–2 to –4 share points**
- Improve Product 2 continuing writer retention by **+5%**
- Increase Product 2 claims per HCP in targeted segments by **+5–10%**


## Geographic Analysis: USA State-Level Market Map

Using HCP state codes from the demographics data, this section maps the injectable anesthesia market across all 50 states. Three views: total market volume by state, Product 2 versus Product 3 share side-by-side, and Product 2 claims year-over-year change (2017→2018) to highlight states with the sharpest erosion.


In [286]:
# ── USA State-Level Market Choropleths ───────────────────────────────────────
# hcp_state = 2-letter state abbreviation from HCP demographics

state_data = (
    market.dropna(subset=['hcp_state'])
    .groupby(['hcp_state', 'market_product'], observed=True)
    .agg(claims=('claim_id', 'nunique'), patients=('patient_id', 'nunique'))
    .reset_index()
)

# ── Map 1: Total market claims by state ───────────────────────────────────────
total_by_state = (
    state_data.groupby('hcp_state')['claims'].sum().reset_index(name='total_claims')
)

fig1 = px.choropleth(
    total_by_state,
    locations='hcp_state',
    locationmode='USA-states',
    color='total_claims',
    scope='usa',
    color_continuous_scale='Blues',
    title='Total Injectable Anesthesia Market Claims by HCP State (2016–2018)',
    labels={'total_claims': 'Total Claims', 'hcp_state': 'State'},
)
fig1.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=450)
fig1.show()

# ── Map 2: Product 2 vs Product 3 share by state (side-by-side) ───────────────
state_wide = state_data.pivot_table(
    index='hcp_state', columns='market_product', values='claims', aggfunc='sum', fill_value=0
).reset_index()
for prod in PRODUCT_ORDER:
    if prod not in state_wide.columns:
        state_wide[prod] = 0

state_wide['total'] = state_wide[PRODUCT_ORDER].sum(axis=1)
state_wide['p2_share'] = (state_wide['Product 2'] / state_wide['total'].replace(0, float('nan'))) * 100
state_wide['p3_share'] = (state_wide['Product 3'] / state_wide['total'].replace(0, float('nan'))) * 100

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Product 2 (Variant) Claim Share %', 'Product 3 (Competitor) Claim Share %'],
    specs=[[{'type': 'choropleth'}, {'type': 'choropleth'}]],
    horizontal_spacing=0.05,
)
fig2.add_trace(
    go.Choropleth(
        locations=state_wide['hcp_state'], z=state_wide['p2_share'],
        locationmode='USA-states', colorscale='Teal',
        colorbar=dict(title='Share %', x=0.45, len=0.85, thickness=12),
        zmin=0, zmax=state_wide[['p2_share', 'p3_share']].max().max(),
        name='P2 Share',
    ), row=1, col=1
)
fig2.add_trace(
    go.Choropleth(
        locations=state_wide['hcp_state'], z=state_wide['p3_share'],
        locationmode='USA-states', colorscale='Reds',
        colorbar=dict(title='Share %', x=1.0, len=0.85, thickness=12),
        zmin=0, zmax=state_wide[['p2_share', 'p3_share']].max().max(),
        name='P3 Share',
    ), row=1, col=2
)
fig2.update_geos(scope='usa')
fig2.update_layout(
    title='Product 2 vs Product 3: Claim Share % by HCP State',
    height=420, margin=dict(l=0, r=0, t=50, b=0),
)
fig2.show()

# ── Map 3: Product 2 YoY claims change 2017→2018 by state ─────────────────────
p2_state_year = (
    market[market['market_product'] == 'Product 2']
    .dropna(subset=['hcp_state'])
    .groupby(['hcp_state', 'claim_year'])['claim_id']
    .nunique()
    .unstack(fill_value=0)
)
p2_state_year.columns = p2_state_year.columns.astype(int)
for yr in [2016, 2017, 2018]:
    if yr not in p2_state_year.columns:
        p2_state_year[yr] = 0

p2_state_year['yoy_pct'] = (
    (p2_state_year[2018] - p2_state_year[2017])
    / p2_state_year[2017].replace(0, float('nan')) * 100
)
p2_yoy = p2_state_year.reset_index()[['hcp_state', 2017, 2018, 'yoy_pct']].dropna(subset=['yoy_pct'])
p2_yoy.columns = ['hcp_state', 'claims_2017', 'claims_2018', 'yoy_pct']

fig3 = px.choropleth(
    p2_yoy,
    locations='hcp_state',
    locationmode='USA-states',
    color='yoy_pct',
    scope='usa',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    title='Product 2 Claims YoY Change % (2017→2018) by HCP State — Red = Declining, Green = Growing',
    labels={'yoy_pct': 'YoY Change %', 'hcp_state': 'State'},
    hover_data={'claims_2017': True, 'claims_2018': True},
)
fig3.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=450)
fig3.show()

print(f'States with Product 2 data: {len(p2_yoy)}')
print('Top 5 declining states:')
print(p2_yoy.sort_values('yoy_pct').head(5)[['hcp_state', 'claims_2017', 'claims_2018', 'yoy_pct']].to_string(index=False))


/var/folders/0s/hrv03yx525sdxdrrygw_bz0h0000gn/T/ipykernel_10383/1431090175.py:30: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  state_wide = state_data.pivot_table(


States with Product 2 data: 58
Top 5 declining states:
hcp_state  claims_2017  claims_2018     yoy_pct
       MA            1            0 -100.000000
       MI            4            0 -100.000000
       NH            6            1  -83.333333
       WA            4            1  -75.000000
       NC            4            1  -75.000000


**Observation and Recommendations — Geographic Analysis: USA State Map**

**Observations:**
- Map 1 confirms that injectable anesthesia market volume is heavily concentrated in a **small number of high-population states** (CA, NY, TX, FL, IL) — these states disproportionately determine national share outcomes.
- Map 2 shows states where **Product 3 share visibly exceeds Product 2 share** — these are active competitive displacement zones that match the territory-level findings from Question 1C.
- Map 3 (YoY change) directly pinpoints states where Product 2 **lost ground from 2017 to 2018** — the red states represent the highest-urgency recovery markets and overlap with the top 5 decline territories (Atlanta/GA, LA–San Diego/CA, New York/NY, Phoenix/AZ, St. Louis/MO).

**Recommendations:**
1. **Layer the state YoY decline map with HCP density** — determine whether drops are driven by a few high-volume writers exiting or by broad market contraction; tailor intervention accordingly.
2. **Concentrate resources in high-volume red states first** — a 2–3% share recovery in CA or NY is worth more in absolute claims volume than a full recovery in a low-volume state.
3. **Use the side-by-side P2 vs P3 map (Map 2) in leadership reviews** — it clearly visualizes the competitive displacement narrative without requiring deep data knowledge.


## Question 2. Key Market Drivers of the Injectable Anesthesia Market

This section investigates patient age, diagnosis specialty, HCP specialty, and new/continuing writer trends for the full injectable anesthesia market across Products 1-4 during 2016-2018.


### Question 2A. Diagnosis Specialty and HCP Specialty Mix

The diagnosis specialty chart uses the diagnosis-code initial mapped from `Diagnosis_Code_Mapping.csv`. The HCP specialty chart counts unique writers from the HCP demographics specialty group.


In [287]:
import math
import html


def _safe_label(value):
    return html.escape(str(value))


def pie_svg(series, title):
    data = series.dropna()
    total = data.sum()
    width, height = 900, 430
    cx, cy, radius = 230, 220, 145
    colors = ['#1B2A4A', '#2D4A7A', '#4A6FA5', '#6B8BB5', '#8BAAC5']
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="40" y="34" font-size="20" font-weight="700" fill="#111827">{_safe_label(title)}</text>',
    ]
    start_angle = -90
    for i, (label, value) in enumerate(data.items()):
        pct = 0 if total == 0 else value / total
        end_angle = start_angle + pct * 360
        x1 = cx + radius * math.cos(math.radians(start_angle))
        y1 = cy + radius * math.sin(math.radians(start_angle))
        x2 = cx + radius * math.cos(math.radians(end_angle))
        y2 = cy + radius * math.sin(math.radians(end_angle))
        large_arc = 1 if pct > 0.5 else 0
        color = colors[i % len(colors)]
        parts.append(
            f'<path d="M {cx} {cy} L {x1:.2f} {y1:.2f} A {radius} {radius} 0 {large_arc} 1 {x2:.2f} {y2:.2f} Z" fill="{color}"/>'
        )
        mid_angle = start_angle + pct * 180
        tx = cx + (radius * 0.65) * math.cos(math.radians(mid_angle))
        ty = cy + (radius * 0.65) * math.sin(math.radians(mid_angle))
        if pct >= 0.06:
            parts.append(f'<text x="{tx:.1f}" y="{ty:.1f}" text-anchor="middle" font-size="12" fill="white" font-weight="700">{pct*100:.1f}%</text>')
        ly = 95 + i * 44
        parts.append(f'<rect x="470" y="{ly-13}" width="14" height="14" fill="{color}"/>')
        parts.append(f'<text x="494" y="{ly}" font-size="13" fill="#374151">{_safe_label(label)}: {_fmt_number(value)} ({pct*100:.1f}%)</text>')
        start_angle = end_angle
    parts.append('</svg>')
    display(HTML(''.join(parts)))


def horizontal_bar_svg(series, title, x_label):
    data = series.dropna().astype(float)
    width = 900
    row_h = 34
    height = max(260, 110 + row_h * len(data))
    margin_l, margin_r, margin_t, margin_b = 250, 50, 68, 46
    plot_w = width - margin_l - margin_r
    max_val = max(1, data.max())
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="40" y="34" font-size="20" font-weight="700" fill="#111827">{_safe_label(title)}</text>',
        f'<text x="{margin_l}" y="{height-12}" font-size="12" fill="#6B7280">{_safe_label(x_label)}</text>',
    ]
    for i, (label, value) in enumerate(data.items()):
        y = margin_t + i * row_h
        bar_w = plot_w * value / max_val
        parts.append(f'<text x="{margin_l-14}" y="{y+17}" text-anchor="end" font-size="12" fill="#374151">{_safe_label(label)}</text>')
        parts.append(f'<rect x="{margin_l}" y="{y}" width="{bar_w:.1f}" height="22" fill="#1B2A4A"/>')
        parts.append(f'<text x="{margin_l+bar_w+8}" y="{y+16}" font-size="12" fill="#111827">{_fmt_number(value)}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))


def vertical_bar_svg(series, title, y_label, color='#1B2A4A'):
    data = series.astype(float)
    width, height = 900, 430
    margin_l, margin_r, margin_t, margin_b = 76, 36, 68, 82
    plot_w = width - margin_l - margin_r
    plot_h = height - margin_t - margin_b
    max_val = max(1, data.max())
    slot_w = plot_w / len(data)
    bar_w = min(58, slot_w * 0.62)
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="34" font-size="20" font-weight="700" fill="#111827">{_safe_label(title)}</text>',
        f'<text x="20" y="{margin_t+plot_h/2}" transform="rotate(-90 20 {margin_t+plot_h/2})" text-anchor="middle" font-size="12" fill="#6B7280">{_safe_label(y_label)}</text>',
    ]
    for frac in [0, 0.25, 0.5, 0.75, 1.0]:
        y = margin_t + plot_h * (1 - frac)
        parts.append(f'<line x1="{margin_l}" y1="{y}" x2="{margin_l+plot_w}" y2="{y}" stroke="#E5E7EB"/>')
        parts.append(f'<text x="{margin_l-8}" y="{y+4}" text-anchor="end" font-size="11" fill="#6B7280">{_fmt_number(max_val*frac)}</text>')
    for i, (label, value) in enumerate(data.items()):
        x = margin_l + i * slot_w + (slot_w - bar_w) / 2
        bar_h = plot_h * value / max_val
        y = margin_t + plot_h - bar_h
        parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" fill="{color}"/>')
        parts.append(f'<text x="{x+bar_w/2:.1f}" y="{y-7:.1f}" text-anchor="middle" font-size="11" fill="#111827">{_fmt_number(value)}</text>')
        parts.append(f'<text x="{x+bar_w/2:.1f}" y="{height-46}" text-anchor="end" transform="rotate(-35 {x+bar_w/2:.1f} {height-46})" font-size="11" fill="#374151">{_safe_label(label)}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))


def grouped_bar_svg(data, title, y_label, colors=None):
    data = data.astype(float)
    colors = colors or {col: '#1B2A4A' for col in data.columns}
    width, height = 920, 460
    margin_l, margin_r, margin_t, margin_b = 76, 190, 70, 82
    plot_w = width - margin_l - margin_r
    plot_h = height - margin_t - margin_b
    max_val = max(1, data.max().max())
    group_w = plot_w / len(data.index)
    bar_w = min(34, group_w * 0.72 / len(data.columns))
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="34" font-size="20" font-weight="700" fill="#111827">{_safe_label(title)}</text>',
        f'<text x="20" y="{margin_t+plot_h/2}" transform="rotate(-90 20 {margin_t+plot_h/2})" text-anchor="middle" font-size="12" fill="#6B7280">{_safe_label(y_label)}</text>',
    ]
    for frac in [0, 0.25, 0.5, 0.75, 1.0]:
        y = margin_t + plot_h * (1 - frac)
        parts.append(f'<line x1="{margin_l}" y1="{y}" x2="{margin_l+plot_w}" y2="{y}" stroke="#E5E7EB"/>')
        parts.append(f'<text x="{margin_l-8}" y="{y+4}" text-anchor="end" font-size="11" fill="#6B7280">{_fmt_number(max_val*frac)}</text>')
    for gi, idx in enumerate(data.index):
        start_x = margin_l + gi * group_w + (group_w - bar_w * len(data.columns)) / 2
        parts.append(f'<text x="{margin_l+gi*group_w+group_w/2}" y="{height-42}" text-anchor="middle" font-size="12" fill="#374151">{_safe_label(idx)}</text>')
        for ci, col in enumerate(data.columns):
            value = data.loc[idx, col]
            x = start_x + ci * bar_w
            bar_h = plot_h * value / max_val
            y = margin_t + plot_h - bar_h
            parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w-3:.1f}" height="{bar_h:.1f}" fill="{colors.get(col, "#1B2A4A")}"/>')
    for i, col in enumerate(data.columns):
        y = 92 + i * 24
        parts.append(f'<rect x="{width-margin_r+28}" y="{y-12}" width="12" height="12" fill="{colors.get(col, "#1B2A4A")}"/>')
        parts.append(f'<text x="{width-margin_r+46}" y="{y}" font-size="12" fill="#374151">{_safe_label(col)}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))


def multi_line_svg(data, title, y_label, colors=None):
    data = data.astype(float)
    colors = colors or {col: '#1B2A4A' for col in data.columns}
    width, height = 920, 430
    margin_l, margin_r, margin_t, margin_b = 80, 180, 68, 68
    plot_w = width - margin_l - margin_r
    plot_h = height - margin_t - margin_b
    max_val = max(1, data.max().max()) * 1.15
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_l}" y="34" font-size="20" font-weight="700" fill="#111827">{_safe_label(title)}</text>',
        f'<text x="20" y="{margin_t+plot_h/2}" transform="rotate(-90 20 {margin_t+plot_h/2})" text-anchor="middle" font-size="12" fill="#6B7280">{_safe_label(y_label)}</text>',
    ]
    for frac in [0, 0.25, 0.5, 0.75, 1.0]:
        y = margin_t + plot_h * (1 - frac)
        parts.append(f'<line x1="{margin_l}" y1="{y}" x2="{margin_l+plot_w}" y2="{y}" stroke="#E5E7EB"/>')
        parts.append(f'<text x="{margin_l-8}" y="{y+4}" text-anchor="end" font-size="11" fill="#6B7280">{_fmt_number(max_val*frac)}</text>')
    x_positions = {idx: margin_l + plot_w * i / max(1, len(data.index)-1) for i, idx in enumerate(data.index)}
    for idx, x in x_positions.items():
        parts.append(f'<text x="{x}" y="{height-34}" text-anchor="middle" font-size="12" fill="#374151">{_safe_label(idx)}</text>')
    for col in data.columns:
        points = []
        for idx in data.index:
            val = data.loc[idx, col]
            x = x_positions[idx]
            y = margin_t + plot_h * (1 - val / max_val)
            points.append((x, y, val))
        parts.append(f'<polyline points="{" ".join(f"{x},{y}" for x, y, _ in points)}" fill="none" stroke="{colors.get(col, "#1B2A4A")}" stroke-width="3"/>')
        for x, y, val in points:
            parts.append(f'<circle cx="{x}" cy="{y}" r="4" fill="{colors.get(col, "#1B2A4A")}"/>')
        parts.append(f'<text x="{margin_l+plot_w+18}" y="{points[-1][1]+4}" font-size="12" fill="{colors.get(col, "#1B2A4A")}" font-weight="700">{_safe_label(col)}</text>')
    parts.append('</svg>')
    display(HTML(''.join(parts)))

market_drivers = analysis_ready[
    (analysis_ready['is_market_product_line']) &
    (analysis_ready['claim_year'].isin(YEARS))
].copy()
market_drivers['claim_year'] = market_drivers['claim_year'].astype(int)

# Diagnosis specialty: top 5 by unique claim count.
diag_specialty_claims = (
    market_drivers.dropna(subset=['diagnosis_specialty'])
    .groupby('diagnosis_specialty')['claim_id']
    .nunique()
    .sort_values(ascending=False)
)
top5_diag_specialty = diag_specialty_claims.head(5)

display_table(top5_diag_specialty.to_frame('unique_claims'), 'Top 5 Diagnosis Specialties by Unique Claims', digits=0)
pie_svg(top5_diag_specialty, 'Top 5 Diagnosis Specialties: Share of Market Claims')

# HCP specialty: unique writers by HCP specialty group.
hcp_specialty_writers = (
    market_drivers.dropna(subset=['hcp_specialty'])
    .groupby('hcp_specialty')['hcp_id']
    .nunique()
    .sort_values(ascending=True)
)

display_table(hcp_specialty_writers.to_frame('unique_writers'), 'Unique HCP Writers by Specialty', digits=0)
horizontal_bar_svg(hcp_specialty_writers, 'HCP Specialty Groups by Number of Unique Writers', 'Unique HCP Writers')


,unique_claims
diagnosis_specialty,
Circulatory System,7290
Factors Influencing Health Status and Contact with Health Services,1818
"Symptoms, Signs and Abnormal Clinical and Lab Findings",1281
Musculoskeletal and Connective Tissue,1189
"Endocrine, Nutritional, Metabolic",754


,unique_writers
hcp_specialty,
Neurology,58
Gastroenterology,63
Orthopedics,64
Cardiology,86
Anesthesiology,228


**Observation and Recommendations — Question 2A: Specialty Mix**

**Observations:**
- **Anesthesiology dominates** HCP writer volume — it is the anchor specialty for this market and must be the primary commercial focus.
- The next-tier influencer specialties are **Cardiology, Orthopedics, Gastroenterology, and Neurology** — these represent expansion opportunity once the Anesthesiology base is secured.
- The dominant patient diagnosis specialty is **Diseases of the Circulatory System** — meaning these drugs are most frequently used in cardiovascular procedure contexts, which should shape clinical messaging and case studies.
- Product 2 should not spread commercial effort evenly across all specialties; concentration in Anesthesiology and Cardiology will capture the majority of market volume.

**Recommendations:**
1. **Prioritize Anesthesiology as the conversion anchor for Product 2** — this is where Product 1's existing loyalty can be leveraged and where Product 3 is gaining fastest; specialty-led conversion starts here.
2. **Drive Product 2 recovery via specialty-led conversion in the 5 decline territories** — match territory-level field targeting with Anesthesiology and Cardiology HCP lists in Atlanta, LA–San Diego, New York, Phoenix, and St. Louis.
3. **Build circulatory-system–aligned clinical messaging for Product 2** — the majority of patients are treated in cardiovascular contexts; messaging that speaks to this clinical setting will resonate more than generic positioning.


### Question 2B. Patient Age and Claim Distribution

The age groups follow the requested buckets: 18-30, 31-40, 41-50, 51-60, 61-70, 71-80, and 81+.


In [288]:
age_bins = [18, 30, 40, 50, 60, 70, 80, 200]
age_labels = ['18-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81+']

age_market = market_drivers.copy()
age_market['age_group'] = pd.cut(
    age_market['patient_age'],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
    right=True,
)
age_market = age_market.dropna(subset=['age_group'])

patients_by_age = (
    age_market.groupby('age_group', observed=False)['patient_id']
    .nunique()
    .reindex(age_labels, fill_value=0)
)
claims_by_age = (
    age_market.groupby('age_group', observed=False)['claim_id']
    .nunique()
    .reindex(age_labels, fill_value=0)
)
claim_pct_by_age = claims_by_age / claims_by_age.sum() * 100

age_summary = pd.DataFrame({
    'unique_patients': patients_by_age,
    'unique_claims': claims_by_age,
    'claim_share_%': claim_pct_by_age,
})
display_table(age_summary, 'Patient and Claim Distribution by Age Group')

vertical_bar_svg(patients_by_age, 'Unique Patients by Age Group', 'Unique Patients', color='#59A14F')
vertical_bar_svg(claim_pct_by_age, 'Claim Share by Age Group', 'Share of Unique Claims (%)', color='#F28E2B')


,unique_patients,unique_claims,claim_share_%
age_group,,,
18-30,499,1983,13.0
31-40,412,1599,10.5
41-50,388,1462,9.6
51-60,474,1903,12.5
61-70,778,3219,21.2
71-80,643,2684,17.7
81+,553,2354,15.5


### Product Mix by Patient Age Group

This adds a visual inspired by the third notebook, but uses the cleaned dataset and unique claims. It shows where Product 3 is winning by patient age segment and where Product 2 needs recovery messaging.


In [289]:
age_product_claims = (
    age_market.groupby(['age_group', 'market_product'], observed=False)['claim_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=age_labels, columns=PRODUCT_ORDER, fill_value=0)
)
age_product_share = age_product_claims.div(age_product_claims.sum(axis=1), axis=0) * 100

display_table(age_product_claims, 'Unique Claims by Age Group and Product', digits=0)
display_table(age_product_share, 'Product Share % Within Each Age Group')
stacked_100_svg(age_product_claims, '100% Stacked Bar: Product Mix by Patient Age Group', 'Unique claim share within each age group, 2016-2018')


market_product,Product 1,Product 2,Product 3,Product 4
age_group,,,,
18-30,1326,153,418,89
31-40,1073,131,319,76
41-50,966,103,328,66
51-60,1232,143,432,96
61-70,2115,229,710,168
71-80,1794,214,560,118
81+,1595,175,493,91


market_product,Product 1,Product 2,Product 3,Product 4
age_group,,,,
18-30,66.8,7.7,21.0,4.5
31-40,67.1,8.2,19.9,4.8
41-50,66.0,7.0,22.4,4.5
51-60,64.7,7.5,22.7,5.0
61-70,65.6,7.1,22.0,5.2
71-80,66.8,8.0,20.8,4.4
81+,67.8,7.4,20.9,3.9


**Observation and Recommendations — Question 2B: Patient Age Distribution**

**Observations:**
- Injectable anesthesia demand is **heavily concentrated in patients aged 61 and above** — this is primarily a senior patient market.
- The 61–70 and 71–80 cohorts represent the highest volume; the 81+ group is also significant, confirming a Medicare-dominant patient population.
- Product 1 leads across all age groups but the gap narrows in the senior cohorts, where Product 3 is gaining share fastest — the 61+ segment is the most competitively contested.
- Younger patient cohorts (18–50) represent relatively small volume and are unlikely to drive meaningful short-term market share recovery.

**Recommendations:**
1. **Tailor patient assistance programs to senior patients** — align Product 2 access and affordability programs with Medicare co-pay structures, supplemental insurance, and senior-care formulary pathways.
2. **Focus Product 2 clinical messaging on the 61+ use case** — highlight efficacy, safety profile, and administration convenience in elderly surgical patients; this is where the volume and the competitive battle are both concentrated.
3. **Launch diagnosis-specialty–aligned HCP outreach in senior-skewed procedure contexts** — target HCPs whose patient panels skew older and who operate in cardiovascular and orthopedic procedure settings.


### Question 2C. New and Continuing Writer Trends

New writers are HCPs writing a product for the first time in the 2016-2018 data window. Continuing writers are calculated as total writers minus new writers for the same product-year.


In [290]:
writer_activity = (
    market_drivers.groupby(['market_product', 'claim_year', 'hcp_id'], observed=False)
    .size()
    .reset_index(name='claim_lines')
)

first_writer_year = (
    writer_activity.groupby(['market_product', 'hcp_id'], observed=False)['claim_year']
    .min()
    .reset_index(name='first_writer_year')
)

writer_activity = writer_activity.merge(first_writer_year, on=['market_product', 'hcp_id'], how='left')
writer_activity['is_new_writer'] = writer_activity['claim_year'] == writer_activity['first_writer_year']

new_writers = (
    writer_activity[writer_activity['is_new_writer']]
    .groupby(['claim_year', 'market_product'], observed=False)['hcp_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=YEARS, columns=PRODUCT_ORDER, fill_value=0)
)

total_writers = (
    writer_activity.groupby(['claim_year', 'market_product'], observed=False)['hcp_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=YEARS, columns=PRODUCT_ORDER, fill_value=0)
)

continuing_writers = total_writers - new_writers

writer_trend_summary = pd.concat(
    {'total_writers': total_writers, 'new_writers': new_writers, 'continuing_writers': continuing_writers},
    axis=1,
)
display_table(writer_trend_summary, 'New and Continuing Writer Summary', digits=0)

grouped_bar_svg(new_writers, 'New Writers by Product and Year', 'New HCP Writers', PRODUCT_COLORS)
multi_line_svg(continuing_writers, 'Continuing Writers by Product', 'Continuing HCP Writers', PRODUCT_COLORS)


total_writers                               new_writers                               continuing_writers            \
market_product     Product 1 Product 2 Product 3 Product 4   Product 1 Product 2 Product 3 Product 4          Product 1 Product 2   
claim_year                                                                                                                          
2016                     492       285       344        72         492       285       344        72                  0         0   
2017                     495       276       416       187           7       111       119       152                488       165   
2018                     496       226       468       268           0        34        32       140                496       192   

                                    
market_product Product 3 Product 4  
claim_year                          
2016                   0         0  
2017                 297        35  
2018                 436       128

### Writer Retention Visual

This translates the new/continuing writer analysis into a retention-style view: what percentage of each year's writers were continuing writers rather than first-time writers.


In [291]:
continuing_writer_rate = continuing_writers / total_writers.replace(0, pd.NA) * 100
display_table(continuing_writer_rate, 'Continuing Writer Rate % by Product and Year')
stacked_100_svg(continuing_writers, 'Continuing Writer Mix by Product and Year', 'Share of continuing HCP writers by product')
line_svg(continuing_writer_rate, 'Line Graph: Continuing Writer Rate by Product', 'Continuing writers divided by total writers (%)')


market_product,Product 1,Product 2,Product 3,Product 4
claim_year,,,,
2016,0.0,0.0,0.0,0.0
2017,98.6,59.8,71.4,18.7
2018,100.0,85.0,93.2,47.8


**Observation and Recommendations — Question 2C: New and Continuing Writer Trends**

**Observations:**
- **Product 3 and Product 4 are winning new HCP adoption in 2017–2018** — competitors are growing their writer base faster than Product 2 is acquiring new writers.
- **Product 1 retains the largest continuing writer base** but faces pressure as new-writer growth shifts toward competitors; its loyalty base is eroding at the margins.
- **Product 2 is not serving as the effective "absorber" of Product 1's decline** — it is weak in new HCP acquisition, meaning Product 3 is capturing first-time writers before Product 2 can convert them.
- Share loss is being driven primarily by **new-writer wins going to competitors** — the competitive battle is being lost at the acquisition stage, not only at the retention stage.
- Continuing writer rates show that once a doctor commits to a product they tend to stick with it — this means **early acquisition is critical**, because switching an entrenched Product 3 writer back later is much harder.

**Recommendations:**
1. **Engage new prescribers through tele-detailing, KOL-led webinars, and peer-to-peer programs for Product 2** — make Product 2 the first drug a new anesthesia writer reaches for, not a fallback.
2. **Create separate programs for new-writer acquisition and continuing-writer retention** — they require different messages: new writers need clinical confidence, continuing writers need reinforcement and depth support.
3. **Prioritize writers who also write Product 3 or recently reduced Product 2 volume** — these switch-risk writers are the highest-value retention targets; deploy rep contact and clinical support proactively before they fully convert to Product 3.


## Question 3. Strategies to Stop Market Share Erosion and Gain Traction for Product 2

### Most Important Market Observations

| Area | Finding | Urgency |
|---|---|---|
| Market share | Product 2 losing claim, patient, and writer share 2016–2018; Product 3 absorbing Product 1's decline | **Critical** |
| Writer productivity | Product 3 gets more claims and patients per writer; gap is widening | **Critical** |
| Territory erosion | Product 2 drops in Atlanta, LA–San Diego, New York, Phoenix, St. Louis while Product 3 grows | **Critical** |
| Specialty concentration | Anesthesiology dominates; Cardiology/Ortho/GI/Neuro are next-tier influencers | **High** |
| Age concentration | 61+ patients drive the majority of market volume — senior and Medicare-aligned market | **High** |
| New-writer acquisition | Competitors winning new writers 2017–2018; Product 2 weak at acquisition stage | **High** |

---

### Recommended Strategic Framework

**Priority 1 — Stop the Competitive Leak (Immediate)**
- Declare Atlanta, LA–San Diego, New York, Phoenix, and St. Louis as **competitive battleground territories**
- Build HCP target lists of: (a) switch-risk Product 2 writers, (b) Product 3 adopters, (c) high-volume writers underindexing on Product 2
- Deploy competitive differentiation messaging vs. Product 3 with territory-specific field sequencing (rep + email + KOL webinar)

**Priority 2 — Rebuild Product 2 as the Default Substitute for Product 1**
- Leverage existing Product 1 HCP loyalty: promote Product 2 as the superior, evolved variant of Product 1
- Specialty-specific positioning starting with Anesthesiology (anchor), then Cardiology, Orthopedics, Gastroenterology, Neurology
- Align messaging to the dominant clinical context: circulatory-system diagnoses in patients aged 61+

**Priority 3 — Win New Writers Before Product 3 Entrenches Further**
- Engage new prescribers via tele-detailing, webinars, and KOL-led sessions specifically for Product 2
- Target the 200+ Anesthesiology-specialty HCPs identified in the data who are not currently prescribing any of the 4 market products — this is an untapped acquisition opportunity
- Implement "first-to-second write" reinforcement: follow up within 30 days of first Product 2 prescription to prevent drop-off

**Priority 4 — Protect Depth Among Existing Writers**
- Protect Product 1 continuing writers with service/support and "stay" messaging
- For Product 2, set depth KPIs (claims per writer, patients per writer) and flag writers who decline below threshold for proactive outreach

---

### Conversion and Retention Playbooks

| Playbook | Actions |
|---|---|
| **Conversion (reduce leakage)** | Competitive differentiation vs. Product 3 · Territory-specific field + channel sequencing · Procedure-aligned clinical education |
| **Retention (protect depth)** | "First-to-second write" reinforcement · Continuing writer retention triggers · Product 1 loyalty-to-Product 2 bridge programs |

---

### KPI Targets (Track Monthly)
- Increase Product 2 new writers in top 5 territories by **+10–15%**
- Reduce Product 3 share in top 5 territories by **–2 to –4 share points**
- Improve Product 2 continuing writer retention by **+5%**
- Increase Product 2 claims per HCP in targeted segments by **+5–10%**
- **Governance:** Weekly dashboard review + monthly market-correction decisions (re-allocate resources where leakage spikes)


## Question 4. Data Exploration Opportunities and Data Concerns

### Additional Datasets Required

| Dataset | Purpose | Business Question Addressed |
|---|---|---|
| **DDD (Drug Distribution Data)** | Track gaps in accessibility due to low pharmaceutical shipment volumes from manufacturers/wholesalers to pharmacies, hospitals, clinics | Is Product 2's decline driven by supply chain disruptions rather than prescriber preference? |
| **NPA (National Prescription Audit)** | Prescription volumes and trends across retail, mail-order, and long-term care channels | Validate whether Medicare claims trends reflect total market or are understated due to payer mix |
| **NPA Extended Insights** | Adds Method of Payment, Co-pay Amount, Patient Age, and Gender to NPA | Assess Product 2 pricing competitiveness; evaluate whether cost is driving prescribers toward Product 3 |
| **NPP (Non-Personal Promotion) Dataset** | Digital channel and content engagement by physician | Identify which channels (email, webinars, digital ads) drive Product 2 engagement; optimize spend |
| **Sales Force / Calls Data** | Rep call frequency, reach, and message exposure by territory and HCP | Determine if Product 2 erosion correlates with sales-force coverage gaps in the 5 decline territories |
| **Formulary / Access / Contracting Data** | Payer access, reimbursement status, formulary tier by geography | Determine if Product 2 decline is access-driven (payer restrictions) vs. preference-driven |

---

### Key Data Gaps in the Current Dataset

1. **No promotion or sales activity data** — cannot determine if Product 2 erosion in specific territories is due to under-promotion or competitive over-promotion by Product 3.
2. **No cost or reimbursement detail at the product level** — the claims data contains `clm_line_alowd_chrg_amt` (allowable charge), which can be used for a rough cost comparison across the 4 products, but this does not capture true patient out-of-pocket cost or co-pay burden.
3. **No procedure or site-of-care detail** — cannot link product choice to specific surgical procedure types or care settings, which would explain why Product 3 is gaining in specific clinical segments.
4. **Medicare only** — this dataset covers only Medicare patients; the full market picture requires commercial, Medicaid, and cash-pay claims to understand total market dynamics.
5. **No provider affiliation or account-level purchasing data** — individual HCP claims cannot be linked to hospital formulary decisions or group purchasing contracts, which are critical access drivers.

---

### Recommended Additional Analyses

1. **ML-based HCP Segmentation:** Cluster HCPs by specialty (Anesthesiology and Cardiology), geography (top 5 decline territories), and patient demographics (circulatory diagnosis, 61+ age group) to build a prioritized targeting model for the field force.
2. **Product Cost Competitiveness Analysis:** Use `clm_line_alowd_chrg_amt` from claims data to compare allowable charges across all 4 products. If Product 2 is priced higher, deploy co-pay cards and payer assistance; if lower, make cost advantage central to HCP messaging.
3. **NPP Channel Optimization:** Use digital engagement data to identify which content types and channels drive Product 2 first and repeat prescriptions — weight field spend toward highest-converting channel combinations.
4. **Untapped HCP Opportunity:** The data reveals approximately 200 additional Anesthesiology-specialty HCPs who are not currently prescribing any of the 4 market products — this represents a clean new-writer acquisition pipeline with no competitive entrenchment to overcome.
5. **DDD Supply Chain Audit:** Cross-reference Product 2 claim declines in the 5 territories with DDD distribution data to rule out supply-side causes before doubling down on demand-side interventions.
6. **Formulary Win Analysis:** Map Product 2 payer access by territory against claim decline to separate access-driven losses (require contracting action) from preference-driven losses (require clinical/promotional action).
